# <font color="#418FDE" size="6.5" uppercase>**Estimator Interface**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Explain the roles of estimators, predictors, transformers, and meta-estimators. 
- Use fit, predict, transform, fit_transform, and score appropriately. 
- Inspect and update estimator parameters without confusing them with learned attributes. 


## **1. API Roles**

### **1.1. Estimator Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_01_01.jpg?v=1783946811" width="250">



>* Estimator defines a configurable learning method
>* Fitting lets it learn from data

>* Estimators include models and preprocessing tools
>* Shared interface supports consistent data workflows

>* Parameters are chosen before fitting
>* Learned attributes appear after fitting



### **1.2. Predictors and Transformers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_01_02.jpg?v=1783946812" width="250">



>* Predictors learn patterns from training data
>* They output predictions for new cases

>* Transformers reshape data for later workflow steps
>* They learn patterns but do not predict

>* Transformers prepare raw data for modeling
>* Predictors use prepared features for outputs



### **1.3. Meta Estimator Wrappers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_01_03.jpg?v=1783946814" width="250">



>* Wrappers coordinate one or more estimators
>* Base estimators learn; wrappers guide usage

>* Wrappers reuse one consistent estimator interface
>* They support selection, ensembles, and features

>* Separate wrapper settings from estimator settings
>* Wrappers organize estimators into stronger workflows



## **2. Core Estimator Methods**

### **2.1. Fitting the Model**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_02_01.jpg?v=1783946823" width="250">



>* Fit lets estimators learn from training data
>* Models and transformers adapt to datasets

>* Parameters set behavior before learning
>* Fitting creates data-specific learned attributes

>* Fit only on training data
>* Prevent leakage for realistic evaluation



In [ ]:
#@title Python Code - Fitting the Model

# Demonstrate fitting with a tiny estimator.
# Parameters exist before any learning.
# Learned attributes appear after fitting.

import math

# Define a tiny nearest neighbor estimator.
class TinyNearestNeighbor:

    # Store configuration parameters before fitting.
    def __init__(self, distance_unit="meters"):
        self.distance_unit = distance_unit

    # Learn by storing training examples.
    def fit(self, X, y):
        if len(X) != len(y):
            raise ValueError("X and y lengths differ")

        self.X_train_ = list(X)
        self.y_train_ = list(y)
        self.classes_ = sorted(set(y))
        return self

    # Predict using learned training examples.
    def predict(self, X):
        if not hasattr(self, "X_train_"):
            raise RuntimeError("Call fit before predict")

        predictions = []
        for row in X:
            best_label = self._nearest_label(row)
            predictions.append(best_label)

        return predictions

    # Compute simple accuracy on labeled data.
    def score(self, X, y):
        predictions = self.predict(X)
        correct = sum(a == b for a, b in zip(predictions, y))
        return correct / len(y)

    # Find the closest stored training row.
    def _nearest_label(self, row):
        distances = []
        for train_row, label in zip(self.X_train_, self.y_train_):
            squared = sum((a - b) ** 2 for a, b in zip(row, train_row))
            distances.append((math.sqrt(squared), label))

        return min(distances)[1]

# Create tiny height and weight data.
X_train = [[170, 65], [180, 80], [155, 50], [190, 95]]
y_train = ["medium", "large", "small", "large"]
X_test = [[160, 55], [185, 88]]
y_test = ["small", "large"]

# Instantiate with a parameter only.
model = TinyNearestNeighbor(distance_unit="centimeters")
print("Before fit, parameters:", model.distance_unit)
print("Before fit, learned data exists:", hasattr(model, "X_train_"))

# Fit learns attributes from training data.
model.fit(X_train, y_train)
print("After fit, learned classes:", model.classes_)
print("After fit, stored rows:", len(model.X_train_))

# Predict and score use learned attributes.
predictions = model.predict(X_test)
accuracy = model.score(X_test, y_test)
print("Predictions for new people:", predictions)
print("Test accuracy:", accuracy)



### **2.2. Predict and Score**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_02_02.jpg?v=1783946827" width="250">



>* Predict only after fitting the estimator
>* Use matching features for new data

>* Score summarizes fitted model performance
>* Choose metrics suited to the problem

>* Keep training and evaluation data separate
>* Predict outputs, then score checked results



In [ ]:
#@title Python Code - Predict and Score

# Predict uses patterns learned during fitting.
# Score checks predictions against known answers.
# Separate test data gives fairer evaluation.

# Import small tools from built-in Python.
from statistics import mean

# Store tiny training examples with labels.
train_heights = [58, 62, 65, 70, 74, 78]
train_labels = ["short", "short", "short", "tall", "tall", "tall"]

# Store separate test examples with answers.
test_heights = [60, 67, 72, 76]
test_labels = ["short", "short", "tall", "tall"]

# Define a tiny estimator-like classifier.
class HeightClassifier:

    # Fit learns a threshold from training data.
    def fit(self, heights, labels):
        assert len(heights) == len(labels)
        short_values = [h for h, y in zip(heights, labels) if y == "short"]
        tall_values = [h for h, y in zip(heights, labels) if y == "tall"]

        # Learned attributes usually end with underscores.
        self.threshold_ = (max(short_values) + min(tall_values)) / 2
        return self

    # Predict uses the learned threshold.
    def predict(self, heights):
        assert hasattr(self, "threshold_")
        return ["tall" if h >= self.threshold_ else "short" for h in heights]

    # Score compares predictions with true labels.
    def score(self, heights, labels):
        predictions = self.predict(heights)
        correct = [p == y for p, y in zip(predictions, labels)]
        return mean(correct)

# Create and fit the estimator.
model = HeightClassifier()
model.fit(train_heights, train_labels)

# Predict labels for unseen test heights.
predictions = model.predict(test_heights)
accuracy = model.score(test_heights, test_labels)

# Print a compact teaching summary.
print("Learned threshold:", model.threshold_, "inches")
print("Test heights:", test_heights)
print("Predictions:", predictions)
print("True labels:", test_labels)
print("Test accuracy:", accuracy)



### **2.3. Transform Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_02_03.jpg?v=1783946829" width="250">



>* Transform converts data into new representations
>* It applies rules learned during fit

>* Fit preprocessing only on training data
>* Transform later data using learned settings

>* Use fit_transform on training transformer data
>* Use transform for already-fitted pipeline steps



In [ ]:
#@title Python Code - Transform Methods

# Demonstrate transformer methods without external machine learning libraries.
# Fit learns settings, while transform applies them.
# Fit transform combines both steps for training data.

import numpy as np

# Create tiny training and future datasets.
train_heights = np.array([[60.0], [64.0], [68.0], [72.0]])
future_heights = np.array([[62.0], [70.0]])

# Define a minimal standard scaling transformer.
class SimpleStandardScaler:
    # Fit stores learned training statistics.
    def fit(self, data):
        data = np.asarray(data, dtype=float)
        self.mean_ = data.mean(axis=0)

        self.scale_ = data.std(axis=0)
        self.scale_[self.scale_ == 0] = 1.0
        return self

    # Transform reuses learned statistics only.
    def transform(self, data):
        data = np.asarray(data, dtype=float)
        return (data - self.mean_) / self.scale_

    # Fit transform is convenient for training data.
    def fit_transform(self, data):
        return self.fit(data).transform(data)

# Fit transform learns from training data.
scaler = SimpleStandardScaler()
train_scaled = scaler.fit_transform(train_heights)

# Transform applies the same learned settings.
future_scaled = scaler.transform(future_heights)

# Validate the transformed shapes are unchanged.
assert train_scaled.shape == train_heights.shape
assert future_scaled.shape == future_heights.shape

# Print concise teaching results.
print("Learned mean inches:", round(float(scaler.mean_[0]), 2))
print("Learned spread inches:", round(float(scaler.scale_[0]), 2))
print("First scaled training value:", round(float(train_scaled[0, 0]), 2))
print("First scaled future value:", round(float(future_scaled[0, 0]), 2))
print("Future data was transformed, not refitted.")



## **3. Estimator Parameters**

### **3.1. Parameters Versus Attributes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_03_01.jpg?v=1783946816" width="250">



>* Parameters set behavior before training
>* Attributes store what training learned

>* Parameters guide how estimators process data
>* Attributes store results learned during fitting

>* Parameters set instructions before fitting
>* Attributes show what fitting learned



In [ ]:
#@title Python Code - Parameters Versus Attributes

# This example separates settings from learned results.
# Parameters exist before fitting any data.
# Attributes appear after fitting the estimator.

import numpy as np

# Define a tiny estimator-like teaching class.
class TinyMeanClassifier:

    # Store user-chosen parameters before fitting.
    def __init__(self, threshold=0.5, label_high="tall"):
        self.threshold = threshold
        self.label_high = label_high

    # Return parameters, not learned attributes.
    def get_params(self):
        return {"threshold": self.threshold, "label_high": self.label_high}

    # Update parameters before future fitting.
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

    # Learn attributes from training data.
    def fit(self, heights_meters):
        values = np.asarray(heights_meters, dtype=float)
        assert values.size > 0, "Need at least one height."
        self.mean_height_ = float(values.mean())
        self.is_fitted_ = True
        return self

    # Predict using parameters and learned attributes.
    def predict(self, heights_meters):
        values = np.asarray(heights_meters, dtype=float)
        cutoff = self.mean_height_ + self.threshold
        return np.where(values >= cutoff, self.label_high, "not tall")

# Create small metric height data.
heights = np.array([1.55, 1.70, 1.82, 1.90])
model = TinyMeanClassifier(threshold=0.05, label_high="above cutoff")

# Inspect parameters before fitting.
print("Parameters before fit:", model.get_params())

# Fit creates learned attributes ending with underscores.
model.fit(heights)
print("Learned mean_height_:", round(model.mean_height_, 2))

# Predictions use both parameters and learned attributes.
print("Prediction for 1.85 meters:", model.predict([1.85])[0])

# Change a parameter for future behavior.
model.set_params(threshold=0.15)
print("Updated parameters:", model.get_params())

# The learned attribute remains a fitted result.
print("mean_height_ still learned:", round(model.mean_height_, 2))



### **3.2. Getting Parameters**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_03_02.jpg?v=1783946821" width="250">



>* Parameters show an estimator’s chosen settings
>* They differ from learned fitted attributes

>* View settings across complete workflows
>* Document configurations for reproducible comparisons

>* Parameters show setup, not learned results
>* Use attributes to inspect fitted model outputs



In [ ]:
#@title Python Code - Getting Parameters

# Estimator parameters are configuration settings.
# Learned attributes appear after fitting.
# This example separates both ideas.

# Define a tiny estimator-like class.
class TinyScaler:

    # Store configuration parameters only.
    def __init__(self, center=True, scale=True):
        self.center = center
        self.scale = scale

    # Return current configuration parameters.
    def get_params(self, deep=True):
        return {"center": self.center, "scale": self.scale}

    # Update configuration parameters safely.
    def set_params(self, **params):
        for name, value in params.items():
            setattr(self, name, value)
        return self

    # Learn attributes from training data.
    def fit(self, values):
        self.mean_ = sum(values) / len(values)
        spread = max(values) - min(values)
        self.scale_ = spread or 1
        return self

    # Transform data using learned attributes.
    def transform(self, values):
        shifted = [x - self.mean_ for x in values]
        return [x / self.scale_ for x in shifted]

# Create small height data in centimeters.
heights_cm = [160, 170, 180]
scaler = TinyScaler(center=True, scale=True)

# Inspect parameters before fitting.
print("Parameters before fit:", scaler.get_params())

# Fit creates learned attributes ending with underscores.
scaler.fit(heights_cm)
print("Learned mean_ after fit:", scaler.mean_)

# Parameters remain configuration, not learned results.
print("Parameters after fit:", scaler.get_params())

# Update one parameter without changing learned attributes.
scaler.set_params(scale=False)
print("Updated parameters:", scaler.get_params())

# Show transformed values from learned attributes.
print("Transformed heights:", scaler.transform(heights_cm))



### **3.3. Updating Parameters**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_A/image_03_03.jpg?v=1783946818" width="250">



>* Parameters set estimator behavior before training
>* Updating changes setup, not learned results

>* Tune parameters for tasks and comparisons
>* Avoid manually changing learned attributes

>* Refit after changing training parameters
>* Keep setup choices separate from learned results



In [ ]:
#@title Python Code - Updating Parameters

# Estimator parameters are setup choices.
# Learned attributes appear after fitting.
# Updating parameters requires careful refitting.

# No extra installations are required.

# Define a tiny estimator-like class.
class TinyMeanRegressor:

    # Store configuration parameters only.
    def __init__(self, offset=0.0):
        self.offset = float(offset)

    # Return current setup parameters.
    def get_params(self):
        return {"offset": self.offset}

    # Update setup parameters safely.
    def set_params(self, **params):
        if "offset" in params:
            self.offset = float(params["offset"])
        return self

    # Learn an attribute from data.
    def fit(self, values):
        values = list(values)
        if len(values) == 0:
            raise ValueError("values must not be empty")
        self.mean_ = sum(values) / len(values)
        return self

    # Predict using learned attribute and parameter.
    def predict(self, count):
        if not hasattr(self, "mean_"):
            raise RuntimeError("fit must run before predict")
        return [self.mean_ + self.offset] * int(count)

# Create small training data.
heights_inches = [64, 66, 68, 70]
model = TinyMeanRegressor(offset=0.0)

# Inspect parameters before fitting.
print("Parameters before fit:", model.get_params())
model.fit(heights_inches)

# Inspect learned attribute after fitting.
print("Learned mean_ after fit:", round(model.mean_, 2))
print("Prediction before update:", model.predict(1)[0])

# Update parameter through the interface.
model.set_params(offset=2.0)
print("Parameters after update:", model.get_params())

# Learned attribute remains unchanged.
print("Learned mean_ still:", round(model.mean_, 2))
print("Prediction after update:", model.predict(1)[0])



# <font color="#418FDE" size="6.5" uppercase>**Estimator Interface**</font>


In this lecture, you learned to:
- Explain the roles of estimators, predictors, transformers, and meta-estimators. 
- Use fit, predict, transform, fit_transform, and score appropriately. 
- Inspect and update estimator parameters without confusing them with learned attributes. 

In the next Lecture (Lecture B), we will go over 'Data Contracts'